# LMC2 #333: süre ölçümü, T4

Bu notebook'u yalnızca T4 GPU ile çalıştır (Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU). Grand Challenge T4 kullanıyor, 5 dakika ve 16 GB sınırı bu donanım için geçerli. Ölçüm en büyük görüntü kenarı olan 3000 px'te, üç smp modeliyle, TTA 1, 4 ve 8 için yapılır. CPU satırı GPU'suz durumu gösterir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob
R = '/content/lmc2-task1-w1'
if not os.path.exists(R):
    !git clone -q https://github.com/neuronautsai/lmc2-task1-w1 {R}
!git -C {R} pull -q && git -C {R} log -1 --oneline
!pip -q install segmentation-models-pytorch==0.6.0 xmltodict
%cd {R}
!nvidia-smi --query-gpu=name,memory.total --format=csv

D = '/content/drive/MyDrive/LMC2'
W5 = f'{D}/outputs/w5'
PAIRS = f'{D}/outputs/dup_check/pairs_excluded.csv'
SMP = [f'{W5}/smp_r34_s42_v2/best.pt', f'{W5}/smp_r34_s123/best.pt', f'{W5}/smp_r34_s456/best.pt']
BASE_HO = f'{W5}/base_heldout_s42/best.pt'
BASE3 = [f'{D}/outputs/w3/base_s{s}/best.pt' for s in (42, 123, 456)]
HELDOUT = f'{W5}/smp_r34_s42_v2/heldout_studies.json'
SIZE = 3000
for p in [PAIRS, *SMP, BASE_HO, *BASE3, HELDOUT]:
    print(os.path.exists(p), p)
    assert os.path.exists(p), f'eksik: {p}'
S_SMP, S_BASE3, S_ALL = ' '.join(SMP), ' '.join(BASE3), ' '.join(SMP + [BASE_HO] + BASE3)

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
!python time_inference.py --checkpoint {S_SMP} --size {SIZE} --tta 1 4 8
!python time_inference.py --checkpoint {SMP[0]} --size {SIZE} --tta 1 --device cpu